In [2]:
import os, sys
import numpy as np
import pandas as pd
import scanpy as sc
import torch
from torch import nn
import PINN
from PINN import reader, models, pl, tl
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from TorchDiffEqPack import odesolve
from torchdiffeq import odeint

os.chdir("/ssd/users/Wergillius/Project/PINN_dynamics")

import matplotlib.pyplot as plt
import seaborn as sns

In [1]:
from torch.utils.data import DataLoader

In [ ]:
# CHANG THIS !!!!!
ckpt_path = "logs/David_cellstates-Cbl_Lpxn_scaled_multiBnch/pde_neighborloss/lightning_logs/version_4/checkpoints/epoch=870-total_loss=24.99072456.ckpt"
model_name = ckpt_path.split("/")[2].replace("_tsense","")
model_class = eval(f"models.{model_name}")

In [ ]:
pde_model = model_class.load_from_checkpoint(ckpt_path)

# load data

In [ ]:
# MODEL CLASS and define model
pde_model = model_class.load_from_checkpoint(ckpt_path)
device = pde_model.device

data_name = ckpt_path.split("/")[1].split("-")[0]
adata = sc.read_h5ad(f"data/{data_name}.h5ad")
timepoints = adata.uns['pop']['t']

cellstate_key = ckpt_path.split("/")[1].split("-")[1].replace("_multiBnch", "")

n_timepoint = 8
n_grid = 300

In [ ]:
train_DS = reader.SingleBranch_AnnDS(AnnData=adata, 
                                n_timepoint = n_timepoint,
                                timepoint_key = 'time',
                                cellstate_key=cellstate_key,  #'Actb_Kcnn4_scaled_S'
                                n_grid=n_grid,  
                                log_transform=False,
                                norm_time = False)

def reduce_batchdim(batch):
    # for batch size 1 , remove the batch dimension
    if len(batch) == 1:
        batch = [ts.squeeze(0) for ts in batch[0]]
    return batch
batch_size = 1
train_DL = DataLoader(train_DS, batch_size=batch_size, num_workers=10, shuffle=True, collate_fn=reduce_batchdim)

In [ ]:
train_DS.u_b.shape

In [ ]:
train_DS.t_b.shape

# simulation

In [ ]:
t_list = t_b.detach().clone().flatten() / 5  

        device = s.device

        
        # loss 2 : dynamics 
        # init_condition 


        it = np.random.choice(range(len(t_list)-1))
        t1 = t_list[it+1].item()
        t0 = t_list[it].item()

        step_size = np.around((t1 - t0)/20, decimals=2).item() 
        step_size = step_size if step_size > 0 else 0.05

        step_size = min(step_size, 0.4)

        ut0 = u_b[it,:]
        utp1 = u_b[it+1,:]

        # init condition : ut, s in a square, u in a square
        init_condition = (ut0, s)
        u_int, s_int = odeint(
                    pde_model.ode_func,
                    init_condition,
                    t_list.type(torch.float32).to(device),
                    atol=1e-8,
                    rtol=1e-8,
                    method='midpoint',
                    options = {'step_size': step_size}
                )

        # boundary u of  the next timepoint
        u_int = nn.functional.relu(u_int)